In [ ]:
# Import required packages
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from joblib import Parallel, delayed

In [ ]:
# Directories
CODE_DIR = Path(r"C:\Users\willi\.vscode\Github\ml-from-crowd")
DATA_DIR = Path(r"E:\Research_data\Stocktwits\dataset\v1\data\csv")
FIGURES_DIR = Path(r"C:\Users\willi\.vscode\Github\Figures")
MODEL_DATA_DIR = Path(r"C:\Users\willi\.vscode\Github\Data")

# File names
INPUT_DATA = MODEL_DATA_DIR / "merged_master.pkl"

# Number of CPU cores for parallel processing
N_JOBS = 8

In [3]:
# Load data
data = pd.read_pickle(INPUT_DATA)

In [4]:
# Prepare features and target
TARGET = 'f_cumret1'

# Exclude non-feature columns and potential other targets
NON_FEATURES = ['date', 'ticker', 'permno', 'shrout', 'prc'] 

# Identify numeric columns
numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()

# Filter features
FEATURES = []
for c in numeric_cols:
    if c == TARGET:
        continue
    if c in NON_FEATURES:
        continue
    # Exclude other return/abnormal return columns to prevent leakage
    if c.startswith('ar_'):
        continue
    FEATURES.append(c)

# Remove missing values
model_data = data[[TARGET] + FEATURES].dropna()

print(f"Sample size: {len(model_data):,}")
print(f"Target: {TARGET}")
print(f"Number of features: {len(FEATURES)}")
print(f"Features: {FEATURES}")

Sample size: 16,743,676
Target: f_cumret1
Number of features: 56
Features: ['net_sentiment', 'extreme_bullish_80', 'extreme_bullish_90', 'extreme_bearish_80', 'extreme_bearish_90', 'disagreement_index', 'raw_volume', 'log_volume', 'unique_user_count', 'volume_diff', 'log_volume_change', 'abn_volume_5d', 'abn_attention_std_5d', 'attention_surge_5d', 'abn_volume_21d', 'abn_attention_std_21d', 'attention_surge_21d', 'abn_volume_63d', 'abn_attention_std_63d', 'attention_surge_63d', 'abn_volume_250d', 'abn_attention_std_250d', 'attention_surge_250d', 'silence_gap_hours', 'relative_volume', 'attention_hhi', 'abnormal_sentiment_1d', 'abnormal_sentiment_5d', 'abnormal_sentiment_21d', 'abnormal_sentiment_63d', 'abnormal_sentiment_250d', 'after_hours_volume', 'after_hours_sentiment', 'market_hours_volume', 'market_hours_sentiment', 'midnight_to_morning_volume', 'midnight_to_morning_sentiment', 'pre_market_volume', 'pre_market_sentiment', 'market_open_volume', 'market_open_sentiment', 'late_morni

In [5]:
# Add date column and sort
model_data['date'] = pd.to_datetime(data.loc[model_data.index, 'date'])
model_data = model_data.sort_values('date')
model_data['year_month'] = model_data['date'].dt.to_period('M')

# In-Sample linear regression

In [6]:
# Prepare X and y
X = model_data[FEATURES]
y = model_data[TARGET]

# Fit linear regression model (in-sample)
lr_model = LinearRegression()
lr_model.fit(X, y)

# Make predictions
y_pred = lr_model.predict(X)

# Evaluate performance
r2 = r2_score(y, y_pred)
mse = mean_squared_error(y, y_pred)
rmse = np.sqrt(mse)

print("In-Sample Linear Regression Results")
print("=" * 50)
print(f"R-squared: {r2:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MSE: {mse:.6f}")
print("\nTop 10 Coefficients:")
coef_df = pd.DataFrame({'feature': FEATURES, 'coef': lr_model.coef_})
coef_df['abs_coef'] = coef_df['coef'].abs()
print(coef_df.sort_values('abs_coef', ascending=False).head(10)[['feature', 'coef']])
print(f"  Intercept: {lr_model.intercept_:.6f}")

In-Sample Linear Regression Results
R-squared: 0.000244
RMSE: 0.047090
MSE: 0.002217

Top 10 Coefficients:
                          feature      coef
24                relative_volume -0.055188
3              extreme_bearish_80  0.003127
4              extreme_bearish_90 -0.002054
7                      log_volume -0.001119
29         abnormal_sentiment_63d  0.000845
55  intraday_sentiment_volatility  0.000837
54              holiday_sentiment -0.000689
30        abnormal_sentiment_250d  0.000655
1              extreme_bullish_80 -0.000625
10              log_volume_change -0.000518
  Intercept: 0.001827


# OOS Predictions (Monthly Training, Daily Predictions)

In [7]:
# Parameters
TRAIN_END_DATE = '2011-12-31'
WINDOW = 252  # Rolling window size in trading days

# Get unique dates
unique_dates = model_data['date'].unique()
unique_dates = pd.DatetimeIndex(unique_dates).sort_values()

# Find the starting prediction date (first date after training window)
train_end = pd.to_datetime(TRAIN_END_DATE)
oos_dates = unique_dates[unique_dates > train_end]

# Get unique months in the OOS period
oos_months = model_data.loc[model_data['date'] > train_end, 'year_month'].unique()
oos_months = sorted(oos_months)

print(f"Rolling window: {WINDOW} trading days")
print(f"Training window: {unique_dates[0].strftime('%Y-%m-%d')} to {TRAIN_END_DATE}")
print(f"OOS prediction period: {oos_dates[0].strftime('%Y-%m-%d')} to {oos_dates[-1].strftime('%Y-%m-%d')}")
print(f"Number of OOS dates: {len(oos_dates):,}")
print(f"Number of OOS months: {len(oos_months)}")
print(f"Parallel processing: {N_JOBS} cores")

Rolling window: 252 trading days
Training window: 2010-01-04 to 2011-12-31
OOS prediction period: 2012-01-03 to 2024-12-30
Number of OOS dates: 3,269
Number of OOS months: 156
Parallel processing: 8 cores


In [8]:
def train_and_predict_month(month_idx, pred_month, model_data, unique_dates, oos_dates, FEATURES, TARGET, WINDOW):
    """
    Train linear regression on rolling window and predict for all days in a month.
    Designed for parallel execution with joblib.
    """
    # Get all prediction dates in this month
    month_dates = oos_dates[oos_dates.to_period('M') == pred_month]
    
    if len(month_dates) == 0:
        return []
    
    # Training cutoff: end of the previous month
    first_day_of_month = pred_month.to_timestamp()
    train_cutoff = first_day_of_month - pd.Timedelta(days=2)
    
    # Find the last trading day before or on train_cutoff
    train_dates = unique_dates[unique_dates <= train_cutoff]
    if len(train_dates) == 0:
        return []
    last_train_date = train_dates[-1]
    
    # ROLLING WINDOW: Train on last WINDOW trading days before the month
    last_train_date_idx = unique_dates.get_loc(last_train_date)
    if last_train_date_idx < WINDOW:
        return []
    
    start_date = unique_dates[last_train_date_idx - WINDOW + 1]
    train_mask = (model_data['date'] >= start_date) & (model_data['date'] <= last_train_date)
    X_train = model_data.loc[train_mask, FEATURES]
    y_train = model_data.loc[train_mask, TARGET]
    
    if len(X_train) == 0:
        return []
    
    # Train model
    lr_model = LinearRegression()
    lr_model.fit(X_train, y_train)
    
    # Predict for all days in the month
    month_predictions = []
    for pred_date in month_dates:
        test_mask = model_data['date'] == pred_date
        X_test = model_data.loc[test_mask, FEATURES]
        
        if len(X_test) == 0:
            continue
        
        test_indices = model_data.index[test_mask]
        y_pred = lr_model.predict(X_test)
        
        for idx, pred in zip(test_indices, y_pred):
            month_predictions.append({
                'date': pred_date,
                'index': idx,
                'prediction': pred
            })
    
    return month_predictions

In [9]:
# Run parallel processing across months
print(f"Starting parallel processing with {N_JOBS} jobs...")
print(f"Processing {len(oos_months)} months...")

def get_month_slice(pred_month):
    """Pre-slice model_data to the rows needed for this month's rolling window + predictions,
    so each parallel worker receives a small DataFrame instead of a full copy of model_data."""
    month_dates = oos_dates[oos_dates.to_period('M') == pred_month]
    if len(month_dates) == 0:
        return model_data.iloc[0:0]
    first_day_of_month = pred_month.to_timestamp()
    train_cutoff = first_day_of_month - pd.Timedelta(days=2)
    train_dates = unique_dates[unique_dates <= train_cutoff]
    if len(train_dates) == 0:
        return model_data.iloc[0:0]
    last_train_date = train_dates[-1]
    last_train_date_idx = unique_dates.get_loc(last_train_date)
    if last_train_date_idx < WINDOW:
        return model_data.iloc[0:0]
    start_date = unique_dates[last_train_date_idx - WINDOW + 1]
    end_date = month_dates.max()
    return model_data.loc[(model_data['date'] >= start_date) & (model_data['date'] <= end_date)]

all_results = Parallel(n_jobs=N_JOBS, verbose=10, backend='threading')(
    delayed(train_and_predict_month)(
        month_idx, pred_month, get_month_slice(pred_month), unique_dates, oos_dates,
        FEATURES, TARGET, WINDOW
    )
    for month_idx, pred_month in enumerate(oos_months)
)

# Flatten results
predictions = [pred for month_preds in all_results for pred in month_preds]

print(f"\nCompleted.")
print(f"Total predictions: {len(predictions):,}")

Starting parallel processing with 8 jobs...
Processing 156 months...


[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   2 tasks      | elapsed:  3.2min
[Parallel(n_jobs=8)]: Done   9 tasks      | elapsed:  4.7min
[Parallel(n_jobs=8)]: Done  16 tasks      | elapsed:  5.2min
[Parallel(n_jobs=8)]: Done  25 tasks      | elapsed:  8.2min
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:  9.9min
[Parallel(n_jobs=8)]: Done  45 tasks      | elapsed: 12.1min
[Parallel(n_jobs=8)]: Done  56 tasks      | elapsed: 14.7min
[Parallel(n_jobs=8)]: Done  69 tasks      | elapsed: 18.2min
[Parallel(n_jobs=8)]: Done  82 tasks      | elapsed: 22.4min
[Parallel(n_jobs=8)]: Done  97 tasks      | elapsed: 26.7min
[Parallel(n_jobs=8)]: Done 112 tasks      | elapsed: 30.2min
[Parallel(n_jobs=8)]: Done 129 tasks      | elapsed: 34.3min
[Parallel(n_jobs=8)]: Done 156 out of 156 | elapsed: 39.3min finished



Completed.
Total predictions: 14,545,760


In [10]:
# Convert predictions to DataFrame
predictions_df = pd.DataFrame(predictions)

# Add symbol information
predictions_df = predictions_df.merge(
    data[['permno', 'ticker']].reset_index(),
    left_on='index',
    right_on='index',
    how='left'
)

# Reorder columns
predictions_df = predictions_df[['date', 'permno', 'ticker', 'index', 'prediction']]

# Sort by date and ticker
predictions_df = predictions_df.sort_values(['date', 'ticker']).reset_index(drop=True)

print("Predictions Summary")
print("=" * 50)
print(f"Total observations: {len(predictions_df):,}")
print(f"Non-null predictions: {predictions_df['prediction'].notna().sum():,}")
print(f"\nFirst 10 predictions:")
print(predictions_df.head(10))

Predictions Summary
Total observations: 14,545,760
Non-null predictions: 14,545,760

First 10 predictions:
        date  permno ticker    index  prediction
0 2012-01-03   87432      A  2879304   -0.000015
1 2012-01-03   24643     AA  2377768   -0.000427
2 2012-01-03   12479    AAC  2266267   -0.000146
3 2012-01-03   90020   AACC  2994307   -0.000150
4 2012-01-03   15580   AAME  2343833   -0.000150
5 2012-01-03   10517    AAN  2211052   -0.000162
6 2012-01-03   76868   AAON  2576068   -0.000144
7 2012-01-03   89217    AAP  2947508   -0.000534
8 2012-01-03   14593   AAPL  2339391   -0.014385
9 2012-01-03   90854   AATI  3056541   -0.000156


In [ ]:
# Save predictions with dynamic filename
OUTPUT_FILE = MODEL_DATA_DIR / f"predictions_linear_regression_input={len(FEATURES)}.pkl"
predictions_df.to_pickle(OUTPUT_FILE)

print(f"Predictions saved to: {OUTPUT_FILE}")
print(f"File size: {OUTPUT_FILE.stat().st_size / (1024**2):.2f} MB")
print(f"Shape: {predictions_df.shape}")
print(f"Columns: {list(predictions_df.columns)}")